# Exploring a MODFLOW 6 DISV grid with pyramids-mesh

**MODFLOW 6** is the USGS modular groundwater-flow model — the standard simulator for how water moves through aquifers. It discretises the subsurface in one of a few ways; **DISV** (*Discretisation by Vertices*) is the unstructured one: each model layer is a flat list of polygon cells built from a shared pool of points — a *vertex list* plus a *cell2d* table that names the vertices making up each cell. Because every cell names its own vertices, neighbouring cells can take different shapes and sizes, which is what makes DISV far more flexible than a rigid row/column grid.

The `modflow6_disv` driver reads **one cell layer** of such a grid into a `pyramids.netcdf.ugrid.MeshDataset` — the same 2-D mesh object every pyramids-mesh driver returns. From there the whole pyramids GIS toolbox applies; this package only parses the file.

This sample is the USGS **`ex-gwf-disvmesh`** example (public-domain, shipped with the MODFLOW 6 example suite): a radial groundwater model whose single layer is an unstructured triangular vertex mesh.

## 1. Imports

Everything comes from `pyramids` and the packages it already brings along. `MeshDataset` is the object the driver returns; `add_tiles` (cleopatra) would drape a web-map basemap under a *georeferenced* mesh. Importing from `pyramids` also puts its vendored GDAL/PROJ data in place, so import order is nothing to worry about in a notebook — just import the names you use.

In [ ]:
from pathlib import Path
from pyramids.netcdf.ugrid import MeshDataset
import numpy as np
import pandas as pd
from cleopatra.basemap.tiles import add_tiles

## 2. Locate the example data

This notebook lives in `docs/examples/real-data/`, so the repo-root `examples/data/real-data/` folder is three levels up. Paths inside a notebook are written **relative to the notebook itself**, and `read_file` wants a `pathlib.Path`, never `str(...)`. The grid file sits in its own extracted model directory next to the other MODFLOW input files (`.nam`, `.npf`, `.tdis`, ...), which the reader leaves in place.

In [ ]:
DATA = Path("../../../examples/data/real-data")
mesh_file = DATA / "modflow6" / "ex-gwf-disvmesh" / "ex-gwf-disvmesh.disv"
mesh_file, mesh_file.exists()

## 3. Read the grid

`MeshDataset.read_file(path, driver=...)` parses the DISV file and returns a `MeshDataset` — identical in type to what pyramids produces from a NetCDF UGRID file. We name the `"modflow6_disv"` driver explicitly and pass the **`Path`** object.

In [ ]:
ds = MeshDataset.read_file(mesh_file, driver="modflow6_disv")
type(ds).__name__

The read emits a `UserWarning` that thousands of clockwise faces were reversed to counter-clockwise — that is pyramids normalising every cell to a consistent winding order on the way in, not an error. The mesh is now in hand.

## 4. Let cleopatra pick the plotting backend

That first read switched matplotlib to a headless backend. One call tells cleopatra to detect the notebook and select the inline backend, so every plot from here on embeds directly in the cell. In a plain script or on CI the same call correctly falls back to a non-interactive backend.

In [ ]:
from cleopatra.config import Config
Config.set_matplotlib_backend()

## 5. Explore the dataset

A `MeshDataset` exposes summary properties: `n_node` / `n_face` (mesh size), `crs` / `epsg` (georeferencing, if the format recorded any), `bounds` (the `(minx, miny, maxx, maxy)` extent) and `data_variable_names` (attached field variables).

In [ ]:
pd.Series({
    "n_node": ds.n_node,
    "n_face": ds.n_face,
    "crs": ds.crs,
    "epsg": ds.epsg,
    "bounds": tuple(round(float(b), 1) for b in ds.bounds),
    "data_variable_names": ds.data_variable_names,
})

The layer has **2,778 vertices** and **5,240 cells**. `crs`/`epsg` are `None`: MODFLOW works in **model coordinates** with no georeferencing — the extent runs roughly -1525 -> +1525 in model length units (metres here), a ~3 km square domain centred on the origin. `data_variable_names` is empty because a grid file is pure topology; simulated heads and budgets live in separate MODFLOW output files, not in the grid definition.

## 6. The cell polygons

`ds.mesh.face_node_connectivity.data` is the `(n_face, k)` table of vertex indices per cell, where `k` is the width the widest polygon needs. Two complementary numbers describe a cell: how many non-fill slots its row uses, and how many *distinct* vertices it actually references.

In [ ]:
conn = ds.mesh.face_node_connectivity.data
pd.Series({
    "connectivity_shape": conn.shape,
    "fill_slots(-1)": int((conn == -1).sum()),
    "slots_per_row (unique)": tuple(sorted(set((conn >= 0).sum(axis=1).tolist()))),
    "distinct_vertices (unique)": tuple(sorted({np.unique(r[r >= 0]).size for r in conn})),
})

The table is `(5240, 4)` with **zero `-1` fill slots**, yet every cell references only **3 distinct vertices**. To see how, look at a few raw rows:

In [ ]:
conn[:5]

Each row is `[a, b, c, a]` — a triangle whose first vertex is repeated to pad it out to the 4-wide table, which is why the raw slot count reads 4 while the true polygon is a triangle. So this grid is entirely **triangular**. The table *width* is set by the widest cell the format allows: a genuinely mixed DISV layer (triangles, quads, pentagons together) would record each cell's real vertices here and `-1`-pad the shorter rows. Read the shape for the widest cell, the distinct-vertex count for what each cell truly is.

## 7. Wireframe: `plot_outline()`

The simplest view needs no data — just the cell edges. `plot_outline()` draws the wireframe through cleopatra and embeds it inline. The `if ds.epsg` guard would add a web-map basemap under a georeferenced mesh; this one has no CRS, so it is skipped. This is the fastest check that the file read in correctly.

In [ ]:
g = ds.plot_outline()
if ds.epsg:
    add_tiles(g.ax, crs=ds.epsg)

A dense triangular tessellation fills the square model domain, with the triangles clearly varying in size across it — the grid is refined in some regions and coarse in others. Seeing it draw confirms the vertex list and cell2d connectivity were parsed correctly.

## 8. A data map: colour the mesh by a field

A *filled* map needs one value per cell. This grid carries geometry only, so we synthesize a field — each cell's planimetric **area** — and attach it with `with_variable(..., location="face")`, which returns a **new** dataset (the original `ds` is untouched). `plot` then colours the mesh by it.

In [ ]:
area = ds.mesh.face_areas
d2 = ds.with_variable("cell_area", area, location="face")
g = d2.plot("cell_area", cmap="viridis", colorbar=True, title="Per-cell area (model units squared)")
if ds.epsg:
    add_tiles(g.ax, crs=ds.epsg)

Cell areas range from about **734 to 2,444** square model units, so the colour map exposes the grid's refinement directly: small (dark) triangles where the model resolves detail, large (bright) ones where it coarsens. This is the generic pyramids data-map path — any per-cell field (hydraulic head, drawdown, recharge) plots exactly the same way.

### A statistical summary

With a field attached, `.summary()` reports per-variable statistics (count, min, max, mean, std).

In [ ]:
d2.summary()

One row, 5,240 values, mean area ~1,394 with a wide spread (std ~276) — the numeric confirmation that this is a non-uniform, refined triangulation rather than a regular grid.

## 9. GIS operations, for free

Because the mesh is now a first-class pyramids object, the pyramids GIS layer applies directly — `pyramids-mesh` itself adds none of this. Each cell below runs one operation and interprets it.

### Faces as a GeoDataFrame

`to_geodataframe()` turns every cell into a polygon row of a geopandas `GeoDataFrame` — the standard vector structure for export to shapefile/GeoPackage or for spatial joins.

In [ ]:
gdf = ds.to_geodataframe()
gdf.shape, gdf.head()

5,240 polygon rows, each a triangle — the grid as ordinary vector geometry, ready to leave the mesh world for the shapely/geopandas ecosystem.

### Bounding polygon

`bounding_polygon()` dissolves the cells into a single shapely polygon of the meshed footprint — the gateway to clipping and spatial joins against other vector layers.

In [ ]:
poly = ds.bounding_polygon()
poly.geom_type, tuple(round(float(b), 1) for b in poly.bounds)

A single polygon whose bounds match the dataset extent: the outer edge of the active model domain, with the interior triangulation dissolved away.

### Cell adjacency and connected components

`sparse_adjacency()` is the cell-to-cell adjacency as a SciPy sparse matrix (`nnz` counts the directed shared-edge links). `connected_components()` labels which cells are mutually reachable through that adjacency.

In [ ]:
adj = ds.sparse_adjacency()
labels = ds.connected_components()
{"adjacency_shape": adj.shape, "nnz": int(adj.nnz),
 "n_components": int(np.unique(labels).size)}

A **single connected component**: every cell is reachable from every other through shared edges, as expected for one contiguous model layer. `nnz` is twice the number of internal shared edges (each counted from both neighbours).

### Crop to a window

`crop(bbox=(xmin, ymin, xmax, ymax))` keeps only the cells inside a window and returns a new `MeshDataset`. We take the centred 60% box and plot the result — a clean clip that stays entirely in the `MeshDataset` world.

In [ ]:
b = ds.bounds
cx, cy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2
hx, hy = (b[2] - b[0]) * 0.3, (b[3] - b[1]) * 0.3
sub = ds.crop(bbox=(cx - hx, cy - hy, cx + hx, cy + hy))
{"full_n_face": ds.n_face, "subset_n_face": sub.n_face}

In [ ]:
g = sub.plot_outline()
if ds.epsg:
    add_tiles(g.ax, crs=ds.epsg)

The window keeps the central block of triangles and drops the rim — a quick way to carve a region of interest out of a MODFLOW domain without leaving pyramids.

Because this format records **no CRS**, coordinate-aware steps have nothing to work from: `to_crs(...)` needs a source CRS to reproject from, and `rasterize_like(...)` / `regrid(...)` burn the mesh onto a target raster grid (the `pyramids-gis` raster layer's job). In practice you would first assign the CRS you know the model used, then reproject; here we only name them.

## 10. Takeaway

- **DISV = vertex grid**: a layer is polygon cells built from a shared vertex pool, read straight into a `MeshDataset`.
- The `ex-gwf-disvmesh` layer is **all-triangular** (2,778 vertices, 5,240 cells); each triangle is stored in a 4-wide connectivity table by repeating its first vertex, with no `-1` fill — the format still supports genuinely mixed n-gons.
- It carries **no CRS** (model coordinates), so reprojection/rasterisation need a CRS you assign first; cell areas span ~734-2,444 units squared, exposing the grid's refinement.
- Once read, explore (`n_node`, `bounds`, `summary()`), plot (`plot_outline`, `plot`) and run GIS (`to_geodataframe`, `bounding_polygon`, `sparse_adjacency`, `connected_components`, `crop`) — all from pyramids, not this package.